# 평가지표 설명
1. 독립 평가: Retrieval (검색 성능) - 답안지 기반 질문으로 자동 채점
"RAG가 정답 문서를 똑바로, 그리고 빠르게 찾아오는가?" (자동 채점)

Hit Rate (hits@K): Top-K 안에 정답 문서가 1개라도 있는가? (0 또는 1)

Precision@K: 가져온 문서들 중 진짜 정답의 비율은? (검색의 정확도)

Recall@K: 전체 정답 문서 중, 이번에 찾아온 문서의 비율은? (검색의 꼼꼼함)

MRR (mrr@K): 첫 번째 정답 문서가 몇 등에 노출되었는가? (상위 노출 능력)

2. 독립 평가: Generator (생성 성능) - 답안지 기반 질문으로 자동 채점
"가져온 정보로 얼마나 자연스럽고 훌륭한 문장을 만들어내는가?" (자동 채점)

BLEU Score: 생성된 답변이 정답(Reference)과 '문장 구조 및 단어 순서'가 얼마나 일치하는가? (기계번역 수준의 문장 생성 퀄리티)

ROUGE-L Score: 정답에 있는 '핵심 문맥(가장 긴 공통 단어열)'을 빼먹지 않고 잘 요약해 내었는가?

3. 종단간 평가 (End-to-End) - 답안지 기반 질문으로 자동 채점
"결과적으로 사용자가 원하는 최종 정답을 얻었는가?" (자동 채점)

Token F1 Score: 쓸데없는 말(환각)은 안 하면서도 정답의 핵심 단어들은 모두 말했는가? (정밀도와 재현율의 조화 평균)

LLM-as-a-Judge (확장 예정): GPT-4 등의 채점관이 "문맥상 정답이 맞다"라고 직접 채점 (Answer Relevance 등)

4. 팀원 평가 (Human Evaluation / 운영 로그 평가) - 사람이 질문 던진 내용 기반으로 수동 채점
"실제 사용자의 돌발 질문에 얼마나 찰떡같이 대답하는가?" (수동 채점)

실제 질문 로깅 (log_for_human_eval): 정해진 답안지 없이, 사용자의 진짜 질문과 RAG의 대답을 엑셀에 실시간 누적 저장.

Groundedness (근거 적합성): 팀원이 엑셀을 보고 평가. "RAG가 이 질문에 답하기 위해 찾아온 문서가 적절한가?" (1~5점)

Faithfulness (환각 여부): 팀원이 엑셀을 보고 평가. "찾아온 문서에 있는 내용만으로 대답했는가? 없는 말을 지어내지 않았는가?" (1~5점)

In [ ]:
import os
import json
import collections
import pandas as pd
from typing import List, Dict, Any
from tqdm.auto import tqdm # 로딩 바 출력을 위해 추가

# ---------------------------------------------------------
# [사전 준비] 라이브러리 로드 (필요시 pip install ranx nltk rouge-score)
# ---------------------------------------------------------
from ranx import Qrels, Run, evaluate as ranx_evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# ---------------------------------------------------------
# 0. 가짜 RAG 시스템 (테스트용)
# ---------------------------------------------------------
def dummy_rag_pipeline(question: str) -> Dict[str, Any]:
    """ RAG의 검색(retrieved_ids)과 대답(response)을 모사합니다. """
    return {
        "response": "사업예산은 130,000,000원이며, 계약일로부터 3개월입니다.", 
        "retrieved_ids": ["20241001798", "9999999"],
        # 🌟 사람이 읽고 평가할 수 있도록 '실제 문서 텍스트'도 같이 반환하도록 추가!
        "retrieved_contexts": [
            "[문서 1] 본 사업의 예산은 130,000,000원(VAT 포함)이며, 사업 기간은 계약일로부터 3개월(안정화 1개월 포함)입니다.",
            "[문서 2] 상관없는 다른 공고의 내용입니다..."
        ]
    }

# ---------------------------------------------------------
# 1. 독립 평가: Retrieval (검색 성능)
# ---------------------------------------------------------
def evaluate_retrieval(eval_rows: List[Dict[str, Any]], k: int = 3) -> Dict[str, float]:
    """
    [평가 지표 설명]
    - Hit Rate (hits@K): Top-K 안에 정답 문서가 1개라도 있는가? (0 또는 1)
    - Precision@K: 가져온 K개의 문서 중 정답 문서의 비율은? (정확도)
    - Recall@K: 실제 정답 문서 전체 중, K개 안에 찾아온 문서의 비율은? (재현율)
    - MRR (mrr@K): 첫 번째로 찾은 정답 문서가 몇 등(Rank)에 있는가? (상위 노출 점수)
    """
    qrels_dict, run_dict = {}, {}
    for idx, row in enumerate(eval_rows):
        qid = str(row.get("qid", f"q{idx}"))
        # 정답 문서 세팅
        qrels_dict[qid] = {str(doc_id): 1.0 for doc_id in row.get("gold_ids", [])}
        # 검색된 문서 세팅 (순위에 따라 가중치 부여)
        run_dict[qid] = {str(doc_id): 1.0/(rank+1) for rank, doc_id in enumerate(row.get("retrieved_ids", []))}
        
    metrics = [f"hits@{k}", f"precision@{k}", f"recall@{k}", f"mrr@{k}"]
    return ranx_evaluate(Qrels(qrels_dict), Run(run_dict), metrics=metrics)

# ---------------------------------------------------------
# 2. 독립 평가: Generator (생성 성능) & 종단간 평가 (F1)
# ---------------------------------------------------------
def evaluate_generation(eval_rows: List[Dict[str, Any]]) -> Dict[str, float]:
    """
    [평가 지표 설명]
    - BLEU: 생성 답변이 정답(Reference)과 '문장 구조/단어 순서'가 얼마나 일치하는가? (번역/생성 품질)
    - ROUGE-L: 정답에 있는 핵심 문맥(Longest Common Subsequence)을 얼마나 안 빼먹고 요약/생성했는가?
    - Token F1 Score (종단간 평가): RAG가 뱉은 최종 답변과 정답 간의 단어 교집합(조화평균). 쓸데없는 말은 안 하면서 정답 단어는 다 말했는가?
    """
    smoothie = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    bleu_scores, rouge_scores, f1_scores = [], [], []
    
    for row in eval_rows:
        pred = row.get("response", "")
        gold = row.get("reference", "")
        
        # 1) BLEU Score 계산
        ref_tokens = [nltk.word_tokenize(gold)]
        pred_tokens = nltk.word_tokenize(pred)
        bleu_scores.append(sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smoothie) if gold and pred else 0.0)
        
        # 2) ROUGE-L Score 계산
        rouge_scores.append(scorer.score(gold, pred)['rougeL'].fmeasure if gold and pred else 0.0)
        
        # 3) Token F1 Score 계산
        common = sum((collections.Counter(gold.split()) & collections.Counter(pred.split())).values())
        if len(gold.split()) == 0 or len(pred.split()) == 0:
            f1_scores.append(1.0 if gold == pred else 0.0)
        elif common == 0:
            f1_scores.append(0.0)
        else:
            p = common / len(pred.split())
            r = common / len(gold.split())
            f1_scores.append((2 * p * r) / (p + r))

    return {
        "avg_bleu": sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0,
        "avg_rougeL": sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0.0,
        "avg_token_f1": sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    }

# ---------------------------------------------------------
# 3. 종단간 평가: LLM-as-a-Judge (뼈대)
# ---------------------------------------------------------
def evaluate_llm_as_a_judge(question: str, response: str, reference: str) -> str:
    """
    [평가 지표 설명]
    - LLM-as-a-Judge: 문자 그대로 일치하지 않더라도, "의미상으로 정답을 맞혔는지" GPT-4 등이 직접 채점합니다.
    (이 함수는 추후 LangChain이나 OpenAI API를 연결하여 점수를 1~5점 등으로 반환하도록 구현합니다. 현재는 뼈대입니다.)
    """
    # TODO: OpenAI API 호출 로직 추가 부분
    return "OpenAI 자원이 남을 경우 추가 예정."

# ---------------------------------------------------------
# 팀원 평가: 실제 사용자의 질문과 답변을 엑셀에 한 줄씩 누적 저장
# ---------------------------------------------------------
def log_for_human_eval(question: str, rag_result: Dict[str, Any], output_csv: str = "real_user_eval_sheet.csv"):
    """
    미리 만들어둔 답안지(json) 없이, 실제 던진 질문과 RAG의 답변을 엑셀에 한 줄씩 추가합니다.
    """
    # 1. RAG가 참고한 문서 내용 정리 (리스트면 줄바꿈으로 합치기)
    contexts = rag_result.get("retrieved_contexts", ["(참고 문서 없음)"])
    context_text = "\n\n".join(contexts) if isinstance(contexts, list) else str(contexts)
    
    # 2. 엑셀에 추가할 새로운 데이터 한 줄 세팅
    new_row = {
        "1. 실제 사용자 질문 (Question)": question,
        "2. RAG 참고 문서 (Context)": context_text,
        "3. RAG 최종 답변 (Response)": rag_result.get("response", ""),
        "Groundedness 점수 (1~5)": "", # 팀원 평가용 빈칸
        "Faithfulness 점수 (1~5)": "", # 팀원 평가용 빈칸
        "코멘트": ""
    }
    
    # 3. 기존 엑셀 파일이 있으면 불러오고, 없으면 새로 만들기
    if os.path.exists(output_csv):
        df = pd.read_csv(output_csv)
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        df = pd.DataFrame([new_row])
        
    # 4. 파일 저장 (utf-8-sig로 저장해야 엑셀에서 한글이 안 깨짐)
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"📝 새로운 사용자 질문이 '{output_csv}' 파일에 추가되었습니다! (현재 총 {len(df)}건 누적)")

# ---------------------------------------------------------
# 4. 팀원 평가 (사람 채점용 시트 내보내기)
# ---------------------------------------------------------
def export_human_eval_sheet(eval_rows: List[Dict[str, Any]], output_csv: str = "human_evaluation_sheet.csv"):
    """
    [평가 지표 설명]
    - Groundedness (근거 적합성): RAG가 찾아온 문서 내용(Context)이 사용자의 질문을 대답하기에 적절한가?
    - Faithfulness (환각 여부): RAG의 답변(Response)이 찾아온 문서 내용(Context)을 벗어나지 않았는가?
    """
    human_eval_data = []
    for row in eval_rows:
        # 찾아온 문서 내용이 리스트 형태라면 보기 좋게 줄바꿈으로 합쳐줍니다.
        contexts = row.get("retrieved_contexts", ["(문서 내용 없음)"])
        context_text = "\n\n".join(contexts) if isinstance(contexts, list) else str(contexts)
        
        human_eval_data.append({
            "문항 ID": row.get("qid"),
            "1. 질문 (Question)": row.get("question"),
            "2. RAG가 참고한 문서 내용 (Context)": context_text,
            "3. RAG의 최종 답변 (Response)": row.get("response"),
            "Groundedness 점수 (1~5): 문서가 질문에 답하기 적합한가?": "", # 팀원 평가칸
            "Faithfulness 점수 (1~5): 답변이 주어진 문서에만 기반했는가?": "", # 팀원 평가칸
            "코멘트": ""
        })
        
    # 엑셀에서 한글이 깨지지 않도록 'utf-8-sig'로 저장합니다.
    pd.DataFrame(human_eval_data).to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"📄 팀원 수동 평가용 시트가 '{output_csv}'로 생성되었습니다. 엑셀로 열어서 평가해주세요!")


In [ ]:
# ---------------------------------------------------------
# 5. 파이프라인 통합 실행부 (실제 JSON 연동 버전)
# ---------------------------------------------------------
if __name__ == "__main__":
    # 🌟 실제 생성하셨던 eval_dataset.json의 경로를 맞춰주세요!
    eval_file_path = '../data/processed/eval/eval_dataset.json' 
    
    if not os.path.exists(eval_file_path):
        print(f"❌ 에러: '{eval_file_path}' 파일을 찾을 수 없습니다. 경로를 확인해주세요!")
        exit()

    print(f"📁 '{eval_file_path}' 파일 로드 중...")
    with open(eval_file_path, 'r', encoding='utf-8') as f:
        real_dataset = json.load(f)
        
    print(f"✅ 총 {len(real_dataset)}개의 모의고사 문항을 성공적으로 불러왔습니다!\n")
    
    eval_rows = []
    print("🚀 1. RAG 파이프라인 응답 수집 중...")
    
    # tqdm을 씌워서 진행률 바를 예쁘게 보여줍니다.
    for item in tqdm(real_dataset, desc="가짜 RAG로 답변 생성 중"):
        # 💡 나중에 이 부분을 '진짜 LangChain RAG 함수'로 바꾸시면 됩니다!
        rag_output = dummy_rag_pipeline(item["question"]) 
        
        row = item.copy()
        row["response"] = rag_output["response"]
        row["retrieved_ids"] = rag_output["retrieved_ids"]
        eval_rows.append(row)
        
    print("\n📊 2. 평가 지표 계산 중...\n")
    
    # 평가 실행
    retrieval_scores = evaluate_retrieval(eval_rows, k=3)
    gen_scores = evaluate_generation(eval_rows)
    
    # LLM-as-a-Judge는 전체를 돌리면 API 비용이 발생하므로, 일단 첫 번째 문항만 샘플로 테스트합니다.
    llm_judge_score = evaluate_llm_as_a_judge(eval_rows[0]["question"], eval_rows[0]["response"], eval_rows[0]["reference"])
    
    # 결과 출력
    print("=== 🎯 [1] 독립 평가: Retrieval (검색) ===")
    for metric, score in retrieval_scores.items():
        print(f" - {metric.upper()}: {score:.4f}")
        
    print("\n=== ✍️ [2] 독립 평가: Generator (생성) ===")
    print(f" - BLEU Score: {gen_scores['avg_bleu']:.4f}")
    print(f" - ROUGE-L Score: {gen_scores['avg_rougeL']:.4f}")
    
    print("\n=== 🏁 [3] 종단간 평가 (End-to-End) ===")
    print(f" - Token F1 Score: {gen_scores['avg_token_f1']:.4f}")
    print(f" - LLM-as-a-Judge Score: {llm_judge_score}")
    
    print("\n=== 👥 [4] 팀원 평가 (Human Eval) ===")
    export_human_eval_sheet(eval_rows)

📁 '../data/processed/eval/eval_dataset.json' 파일 로드 중...
✅ 총 391개의 모의고사 문항을 성공적으로 불러왔습니다!

🚀 1. RAG 파이프라인 응답 수집 중...


가짜 RAG로 답변 생성 중: 100%|██████████| 391/391 [00:00<00:00, 644896.92it/s]


📊 2. 평가 지표 계산 중...

=== 🎯 [1] 독립 평가: Retrieval (검색) ===
 - HITS@3: 0.0121
 - PRECISION@3: 0.0040
 - RECALL@3: 0.0121
 - MRR@3: 0.0121

=== ✍️ [2] 독립 평가: Generator (생성) ===
 - BLEU Score: 0.0142
 - ROUGE-L Score: 0.1318

=== 🏁 [3] 종단간 평가 (End-to-End) ===
 - Token F1 Score: 0.0124
 - LLM-as-a-Judge Score: 5.0 / 5.0 (샘플 1건 테스트)

=== 👥 [4] 팀원 평가 (Human Eval) ===
📄 팀원 수동 평가용 시트가 'human_evaluation_sheet.csv'로 생성되었습니다. 엑셀로 열어서 평가해주세요!


In [9]:
# ---------------------------------------------------------
# 사용자 질문 기반 팀원 평가
# ---------------------------------------------------------
if __name__ == "__main__":
    # 답안지에 없는, 내가 지금 궁금한 아무 질문이나 던져봅니다.
    my_real_questions = [
        "이 시스템 구축하면 우리 학교에 뭐가 좋은데?",
        "예산이 2억 넘는 사업이 있어?"
    ]
    
    for q in my_real_questions:
        print(f"\n🗣️ 질문: {q}")
        # 진짜 RAG 시스템 (현재는 가짜 함수)을 통과시킵니다.
        rag_answer = dummy_rag_pipeline(q) 
        
        # 나온 결과를 바로 팀원 평가용 엑셀에 쏙 집어넣습니다!
        log_for_human_eval(question=q, rag_result=rag_answer)


🗣️ 질문: 이 시스템 구축하면 우리 학교에 뭐가 좋은데?
📝 새로운 사용자 질문이 'real_user_eval_sheet.csv' 파일에 추가되었습니다! (현재 총 1건 누적)

🗣️ 질문: 예산이 2억 넘는 사업이 있어?
📝 새로운 사용자 질문이 'real_user_eval_sheet.csv' 파일에 추가되었습니다! (현재 총 2건 누적)
